#### Querying JSON 

In [0]:
%python
dataset_bookstore = "dbfs:/FileStore/data/bookstore"
#spark.conf.set(f"dataset.bookstore", dataset_bookstore)

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/customers-json")
display(files)

In [0]:
SELECT * FROM read_files(
  '${dataset.bookstore}/customers-json',
  format => 'json'
)

In [0]:
SELECT customer_id, email, first_name, last_name
FROM read_files(
  '${dataset.bookstore}/customers-json',
  format => 'json'
)
WHERE customer_id IS NOT NULL

#### Querying text format

In [0]:
SELECT * FROM read_files(
  '${dataset.bookstore}/customers-json',
  format => 'text'
)

#### Querying binaryFile Format

In [0]:
SELECT path, modificationTime, length, content
FROM read_files(
  '${dataset.bookstore}/customers-json',
  format => 'binaryFile'
)


#### Querying CSV 

In [0]:
SELECT * FROM read_files(
  '${dataset.bookstore}/books-csv',
  format => 'csv',
  header => true,
  delimiter => ','
)

In [0]:
SELECT 
  book_id,
  title,
  author,
  CAST(price AS DOUBLE) as price
FROM read_files(
  'dbfs:/FileStore/data/bookstore/books-csv',
  format => 'csv',
  header => true,
  delimiter => ';'
)


#### Limitations of Non-Delta Tables

In [0]:
CREATE TABLE IF NOT EXISTS books_csv
USING CSV
OPTIONS (
  header = "true",
  delimiter = ","
)
LOCATION '${dataset.bookstore}/books-csv'

In [0]:
SELECT * FROM books_csv

#### CTAS Statements

In [0]:
CREATE OR REPLACE TABLE customers_json AS
SELECT * FROM read_files(
  '${dataset.bookstore}/customers-json',
  format => 'json'
)

In [0]:
SELECT * FROM customers_json LIMIT 10

In [0]:
CREATE OR REPLACE TABLE books_clean AS
SELECT 
  book_id,
  title,
  author,
  CAST(price AS DOUBLE) as price,
  current_timestamp() as loaded_at
FROM read_files(
  'dbfs:/FileStore/data/bookstore/books-csv',
  format => 'csv',
  header => true,
  delimiter => ';'
)
WHERE book_id IS NOT NULL

In [0]:
SELECT * FROM books_clean

In [0]:
SELECT 
  *,
  _metadata.file_path,
  _metadata.file_name,
  _metadata.file_size,
  _metadata.file_modification_time
FROM read_files(
  'dbfs:/FileStore/data/bookstore/customers-json',
  format => 'json'
)

In [0]:
SELECT 
  customer_id,
  email,
  profile:first_name as first_name,
  _metadata.file_name
FROM read_files(
  'dbfs:/FileStore/data/bookstore/customers-json',
  format => 'json'
)
WHERE _metadata.file_name LIKE '%export%'

In [0]:
SELECT 
  book_id,
  title,
  author,
  price,
  _metadata.file_path as source_file,
  _metadata.file_modification_time as file_modified_at
FROM read_files(
  'dbfs:/FileStore/data/bookstore/books-csv',
  format => 'csv',
  header => true,
  delimiter => ';'
)

In [0]:
CREATE OR REPLACE TABLE customers_with_metadata AS
SELECT 
  customer_id,
  email,
  profile:first_name as first_name,
  profile:last_name as last_name,
  _metadata.file_name as source_file,
  _metadata.file_modification_time as source_file_timestamp,
  current_timestamp() as ingestion_timestamp
FROM read_files(
  'dbfs:/FileStore/data/bookstore/customers-json',
  format => 'json'
)
WHERE customer_id IS NOT NULL

In [0]:
SELECT * FROM customers_with_metadata